In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/lukajincharadze/pandas-data/noc_regions.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/olympics-data.xlsx
/kaggle/input/datasets/lukajincharadze/pandas-data/results.parquet
/kaggle/input/datasets/lukajincharadze/pandas-data/bios.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/coffee.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/results.csv
/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [13]:
%pip install -q dagshub mlflow

Note: you may need to restart the kernel to use updated packages.


# Dagshub/Mlflow Initialization

In [14]:
import dagshub
dagshub.init(repo_owner='skoba23', repo_name='ML_Assignment_2', mlflow=True)

Initialized MLflow to track repo "skoba23/ML_Assignment_2"

Repository skoba23/ML_Assignment_2 initialized!

In [15]:
import mlflow
import mlflow.sklearn
import os
TRACKING_URI = "https://dagshub.com/skoba23/ML_Assignment_2.mlflow"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("RandomForest_Training")

print('MLflow Tracking URI:', TRACKING_URI)
print('Setup complete!')

MLflow Tracking URI: https://dagshub.com/skoba23/ML_Assignment_2.mlflow
Setup complete!


## Data Loading

In [16]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

df_train = train_transaction.merge(train_identity, on='TransactionID', how='left')
df_test  = test_transaction.merge(test_identity,   on='TransactionID', how='left')

print('Train shape:', df_train.shape)
print('Test shape: ', df_test.shape)
print('Fraud rate: ', round(df_train['isFraud'].mean() * 100, 2), '%')

Train shape: (590540, 434)
Test shape:  (506691, 433)
Fraud rate:  3.5 %


# Cleaning

In [17]:
mlflow.start_run(run_name="RF_Cleaning")
null_threshold = 0.5
null_ratio = df_train.isnull().mean()
drop_cols = null_ratio[null_ratio > null_threshold].index.tolist()
drop_cols = [c for c in drop_cols if c != "isFraud"]

df_train_clean = df_train.drop(columns=drop_cols)
df_test_clean  = df_test.drop(columns=[c for c in drop_cols if c in df_test.columns])


### Outlier Removal

In [18]:
Q1 = df_train_clean["TransactionAmt"].quantile(0.25)
Q3 = df_train_clean["TransactionAmt"].quantile(0.75)
IQR = Q3 - Q1
before = len(df_train_clean)
df_train_clean = df_train_clean[df_train_clean["TransactionAmt"] < Q3 + 3 * IQR]
after = len(df_train_clean)

### Filling null values with median

In [19]:
num_cols = df_train_clean.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != "isFraud"]
df_train_clean[num_cols] = df_train_clean[num_cols].fillna(df_train_clean[num_cols].median())
df_test_clean[num_cols]  = df_test_clean[[c for c in num_cols if c in df_test_clean.columns]].fillna(
                                df_train_clean[[c for c in num_cols if c in df_test_clean.columns]].median())


### One-Hot Encoding

In [20]:
from sklearn.preprocessing import LabelEncoder
cat_cols = df_train_clean.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    df_train_clean[col] = le.fit_transform(df_train_clean[col].astype(str))
    if col in df_test_clean.columns:
        df_test_clean[col] = le.transform(df_test_clean[col].astype(str).map(
            lambda x: x if x in le.classes_ else le.classes_[0]))

In [21]:
mlflow.log_param("null_threshold", null_threshold)
mlflow.log_param("encoding", "LabelEncoder")
mlflow.log_param("dropped_columns", len(drop_cols))
mlflow.log_param("outlier_method", "IQR_3x")
mlflow.log_metric("removed_outliers", before - after)
mlflow.log_metric("remaining_features", df_train_clean.shape[1])

print(f"Dropped {len(drop_cols)} columns, removed {before - after} outliers")

Dropped 214 columns, removed 36422 outliers


In [22]:
mlflow.end_run()

🏃 View run RF_Cleaning at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2/runs/a14ab1ffe3a440d5a79d448b95cc8fa1
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2


# Feature Engineering

In [23]:
with mlflow.start_run(run_name="RF_FeatureEngineering"):
    df_train_clean["TransactionAmt_log"] = np.log1p(df_train_clean["TransactionAmt"])
    df_test_clean["TransactionAmt_log"]  = np.log1p(df_test_clean["TransactionAmt"])

    df_train_clean["Transaction_day"]  = (df_train_clean["TransactionDT"] // (3600 * 24)) % 7
    df_train_clean["Transaction_hour"] = (df_train_clean["TransactionDT"] // 3600) % 24
    df_test_clean["Transaction_day"]   = (df_test_clean["TransactionDT"] // (3600 * 24)) % 7
    df_test_clean["Transaction_hour"]  = (df_test_clean["TransactionDT"] // 3600) % 24

    df_train_clean["card1_addr1"] = (df_train_clean["card1"].astype(str) + "_" + df_train_clean["addr1"].astype(str))
    df_test_clean["card1_addr1"]  = (df_test_clean["card1"].astype(str) + "_" + df_test_clean["addr1"].astype(str))

    le = LabelEncoder()
    df_train_clean["card1_addr1"] = le.fit_transform(df_train_clean["card1_addr1"].astype(str))

    label_map = {cls: idx for idx, cls in enumerate(le.classes_)}
    df_test_clean["card1_addr1"] = (
        df_test_clean["card1_addr1"].astype(str).map(label_map).fillna(0).astype(int)
    )

    fraud_rate = df_train_clean.groupby("card1")["isFraud"].mean()
    df_train_clean["card1_fraud_rate"] = df_train_clean["card1"].map(fraud_rate)
    df_test_clean["card1_fraud_rate"]  = df_test_clean["card1"].map(fraud_rate).fillna(fraud_rate.mean())

    mean_amt = df_train_clean["TransactionAmt"].mean()
    std_amt  = df_train_clean["TransactionAmt"].std()
    df_train_clean["TransactionAmt_zscore"] = (df_train_clean["TransactionAmt"] - mean_amt) / std_amt
    df_test_clean["TransactionAmt_zscore"]  = (df_test_clean["TransactionAmt"] - mean_amt) / std_amt

    mlflow.log_param("new_features", "TransactionAmt_log, Transaction_day, Transaction_hour, card1_addr1, card1_fraud_rate, TransactionAmt_zscore")
    mlflow.log_metric("total_features", df_train_clean.shape[1])
    print("Feature Engineering done:", df_train_clean.shape)

/tmp/ipykernel_57/498108238.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_train_clean["TransactionAmt_log"] = np.log1p(df_train_clean["TransactionAmt"])
/tmp/ipykernel_57/498108238.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test_clean["TransactionAmt_log"]  = np.log1p(df_test_clean["TransactionAmt"])
/tmp/ipykernel_57/498108238.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns a

Feature Engineering done: (554118, 226)
🏃 View run RF_FeatureEngineering at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2/runs/8663e9b6646b4a9a91e64076baad3133
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2


# Feature Selection

In [24]:
from sklearn.feature_selection import SelectKBest, f_classif

Y = df_train_clean["isFraud"]
X = df_train_clean.drop(columns=["isFraud", "TransactionID"])
for col in X.select_dtypes(include="object").columns:
    X[col] = X[col].astype("category").cat.codes
    
with mlflow.start_run(run_name="RF_FeatureSelection"):
    selector = SelectKBest(f_classif, k=40)
    selector.fit(X, Y)
    selected_features = X.columns[selector.get_support()].tolist()

    mlflow.log_param("method", "SelectKBest_f_classif")
    mlflow.log_param("k", 40)
    mlflow.log_metric("selected_features", len(selected_features))
    print(f"Selected {len(selected_features)} features")

X_final = X[selected_features]
X_test_final = df_test_clean[[c for c in selected_features if c in df_test_clean.columns]]


Selected 40 features
🏃 View run RF_FeatureSelection at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2/runs/85eb97b9c517442f985585e240e62264
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2


### Filling missing columns with 0

In [25]:
for col in selected_features:
    if col not in X_test_final.columns:
        X_test_final[col] = 0
X_test_final = X_test_final[selected_features]

# Training

In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

X_tr, X_val, Y_tr, Y_val = train_test_split(X_final, Y, test_size=0.2, random_state=42, stratify=Y)

### Baseline

In [27]:
with mlflow.start_run(run_name="RF_Train_Baseline"):
    pipe1 = Pipeline([
        ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ])
    pipe1.fit(X_tr, Y_tr)
    preds1 = pipe1.predict_proba(X_val)[:, 1]
    auc1 = roc_auc_score(Y_val, preds1)

    mlflow.log_params({"n_estimators": 100, "max_depth": "None"})
    mlflow.log_metric("val_auc", auc1)
    mlflow.sklearn.log_model(pipe1, "model")
    print(f"Baseline AUC: {auc1:.4f}")

2026/05/07 20:17:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 20:17:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Baseline AUC: 0.8796
🏃 View run RF_Train_Baseline at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2/runs/c5f88db0faad4273a7aa336d8755d645
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2


### Max_Depth Increase

In [28]:
with mlflow.start_run(run_name="RF_Train_MaxDepth10"):
    pipe2 = Pipeline([
        ("model", RandomForestClassifier(n_estimators=100, max_depth=10,
                                          random_state=42, n_jobs=-1))
    ])
    pipe2.fit(X_tr, Y_tr)
    preds2 = pipe2.predict_proba(X_val)[:, 1]
    auc2 = roc_auc_score(Y_val, preds2)

    mlflow.log_params({"n_estimators": 100, "max_depth": 10})
    mlflow.log_metric("val_auc", auc2)
    mlflow.sklearn.log_model(pipe2, "model")
    print(f"MaxDepth=10 AUC: {auc2:.4f}")

2026/05/07 20:18:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 20:18:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MaxDepth=10 AUC: 0.8638
🏃 View run RF_Train_MaxDepth10 at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2/runs/9f95f151933a46ef99e3815d675f00a8
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2


### Balanced Class Weight

In [29]:
with mlflow.start_run(run_name="RF_Train_balanced"):
    pipe3 = Pipeline([
        ("model", RandomForestClassifier(n_estimators=200, max_depth=15,
                                          class_weight="balanced",
                                          random_state=42, n_jobs=-1))
    ])
    pipe3.fit(X_tr, Y_tr)
    preds3 = pipe3.predict_proba(X_val)[:, 1]
    auc3 = roc_auc_score(Y_val, preds3)

    mlflow.log_params({"n_estimators": 200, "max_depth": 15, "class_weight": "balanced"})
    mlflow.log_metric("val_auc", auc3)
    mlflow.sklearn.log_model(pipe3, "model")
    print(f"Balanced AUC: {auc3:.4f}")


2026/05/07 20:18:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 20:18:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Balanced AUC: 0.8830
🏃 View run RF_Train_balanced at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2/runs/622878e619474b32aec99a9dc86220fe
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2


### Min_Samples_Leaf

In [30]:
with mlflow.start_run(run_name="RF_MinSamplesLeaf_Balanced"):
    pipe4 = Pipeline([
        ("model", RandomForestClassifier(n_estimators=200, max_depth=15,
                                          min_samples_leaf=20, class_weight="balanced",
                                          random_state=42, n_jobs=-1))
    ])
    pipe4.fit(X_tr, Y_tr)
    preds4 = pipe4.predict_proba(X_val)[:, 1]
    auc4 = roc_auc_score(Y_val, preds4)

    mlflow.log_params({"n_estimators": 200, "max_depth": 15,
                       "min_samples_leaf": 20, "class_weight": "balanced"})
    mlflow.log_metric("val_auc", auc4)
    mlflow.sklearn.log_model(pipe4, "model")
    print(f"MinSamplesLeaf=20 AUC: {auc4:.4f}")

2026/05/07 20:19:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 20:19:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MinSamplesLeaf=20 AUC: 0.8807
🏃 View run RF_MinSamplesLeaf_Balanced at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2/runs/9d7b8a1608cf4db18f2efc9658fa7a24
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2


### Cross Validation

In [31]:
best_pipe = max([(auc1, pipe1), (auc2, pipe2), (auc3, pipe3), (auc4, pipe4)],
                 key=lambda x: x[0])[1]
best_auc  = max(auc1, auc2, auc3, auc4)

with mlflow.start_run(run_name="RF_CrossValidation"):
    cv_scores = cross_val_score(best_pipe, X_final, Y, cv=5, scoring="roc_auc")

    mlflow.log_param("cv_folds", 5)
    mlflow.log_metric("cv_auc_mean", cv_scores.mean())
    mlflow.log_metric("cv_auc_std", cv_scores.std())
    print(f"CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

CV AUC: 0.8482 ± 0.0350
🏃 View run RF_CrossValidation at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2/runs/34d9f7eef9a345c7a2bde73a2755a35a
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2


# Saving Model Registry

In [32]:
with mlflow.start_run(run_name="RF_BestModel_Registry"):
    mlflow.sklearn.log_model(
        best_pipe,
        artifact_path="model",
        registered_model_name="IEEE_Fraud_RandomForest"
    )
    mlflow.log_metric("val_auc", best_auc)
    print(f"Best model saved! AUC: {best_auc:.4f}")

2026/05/07 20:22:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 20:22:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'IEEE_Fraud_RandomForest'.
2026/05/07 20:22:33 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IEEE_Fraud_RandomForest, version 1
Created version '1' of model 'IEEE_Fraud_RandomForest'.


Best model saved! AUC: 0.8830
🏃 View run RF_BestModel_Registry at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2/runs/00ebcc13e9d747ea8088a4121ec994ad
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/2
